# Janome による日本語の形態素解析

## テキストファイルの読み込み

In [ ]:
# テキストファイルの読み込み
src = "sangetsuki.txt"              # 同じフォルダにあるテキストファイルの名前
f = open(src, encoding="utf_8_sig") # 文字コードが UTF BOM付き でない場合は，utf_8_sig を書き換える
text = f.read()
f.close()

text

## 形態素解析

In [ ]:
# 形態素解析
from janome.tokenizer import Tokenizer
import pandas as pd

tokenizer = Tokenizer()
tlist = tokenizer.tokenize(text)
tab = []

for t in tlist:
    if t.extra==None:
        t.extra = (t.part_of_speech, '*', '*', t.surface, t.surface, t.surface)
    tab.append([t.surface] + t.part_of_speech.split(',') + list(t.extra[1:]))

tokens = pd.DataFrame(tab)
tokens.columns = [
    '表層形', '品詞', '品詞細分類1', '品詞細分類2', '品詞細分類3',
    '活用型', '活用形', '原形', '読み', '発音'
    ]
#tokens.to_csv('tokens.csv', index=False, encoding='utf_8_sig')

tokens

# 日本語ワードクラウドの生成

In [ ]:
# 名詞，動詞，形容詞 の抽出
ind = tokens[tokens["品詞"]=="名詞"]
ind = pd.concat([ind, tokens[tokens["品詞"]=="動詞"]])
ind = pd.concat([ind, tokens[tokens["品詞"]=="形容詞"]])
ind

In [ ]:
# 原形のリストを作り，一部の頻出語を除去する
words_list = list(ind["原形"])
rm_words = [
    "ある", "いる", "する", "せる", "なる", "これ", "それ", "あれ", "どれ",
    "この", "その", "あの", "どの", "もの", "こと", "よう", "れる", "の",
    "ない"
    ]
for rw in rm_words:
    words_list = [w for w in words_list if w!=rw]
words = ' '.join(words_list)
words

In [ ]:
# ワードクラウドの生成表示
from wordcloud import WordCloud
import matplotlib.pyplot as plt
wc = WordCloud(font_path="/c:/Windows/Fonts/YuGothM.ttc", width=500, height=500)
wc.generate(words)
plt.figure(dpi=150)     # dpi の数字を大きくすると，大きく表示
plt.imshow(wc)
plt.axis("off")         # 軸目盛りの非表示
plt.show()